# Python Data Validation and Pydantic - Follow-Along Practice

This notebook contains all the practice examples and code snippets from the Pydantic chapter, structured sequentially: Type Hints, Dataclasses, Basic BaseModel schemas, Field constraints, Custom field validators, Nested models, and Pydantic Settings.

### Install Dependencies
Run the following cell to install Pydantic and Pydantic Settings in your notebook environment.

In [ ]:
!pip install pydantic pydantic-settings

## 1. Type Hints
Type hints do not enforce types at runtime, but enable editors to provide autocompletion and static analysis.

In [ ]:
# Basic variables
age: int = 25
name: str = "Alice"
is_active: bool = True

# Function parameter and return type annotations
def calculate_total(price: float, quantity: int) -> float:
    return price * quantity

# Container types and modern Union/Optional syntax
prices: list[float] = [10.99, 5.50, 20.00]
user_roles: dict[str, str] = {"admin": "Alice", "user": "Bob"}
result: int | float = 10.5
middle_name: str | None = None

## 2. Dataclasses
A `@dataclass` automatically generates boilerplate methods like `__init__()`, `__repr__()`, and `__eq__()` but does not validate types at runtime.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    stock: int = 0  # Default value

# Instantiation and nice __repr__ representation
item = Product("Laptop", 999.99, 5)
print(item)  # Product(name='Laptop', price=999.99, stock=5)

# Note: Dataclasses DO NOT enforce types at runtime
bad_item = Product("Phone", "five hundred", "none")
print("Dataclass allows invalid types:", bad_item)

## 3. Basic BaseModel Example (Pydantic)
Inheriting from `pydantic.BaseModel` enables full runtime type validation and automatic coercion.

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True

# Successful Instantiation with Type Coercion
# String "123" is coerced to integer 123
user = User(id="123", username="alice", email="alice@example.com")
print(user)
print("ID type:", type(user.id))  # <class 'int'>

# Validation failure when coercion is impossible
try:
    bad_user = User(id="not-an-int", username="bob", email="bob@example.com")
except ValidationError as e:
    print("Caught expected ValidationError:\n", e)

## 4. Field Constraints
Using `Field` to add constraints (like min/max, string lengths, regex patterns, and descriptions).

In [ ]:
from pydantic import BaseModel, Field

class ProductProfile(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, description="Price must be greater than zero")
    stock: int = Field(default=0, ge=0)

try:
    ProductProfile(name="C", price=-2.50)
except ValidationError as e:
    print(e)

## 5. Custom Field Validators
Using the `@field_validator` decorator to implement complex validation logic and value transformation.

In [ ]:
from pydantic import BaseModel, field_validator

class SignUp(BaseModel):
    username: str
    password: str

    @field_validator("password")
    @classmethod
    def password_must_be_strong(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("Password must be at least 8 characters long")
        if not any(char.isdigit() for char in v):
            raise ValueError("Password must contain at least one digit")
        return v

try:
    SignUp(username="coder", password="secret")
except ValidationError as e:
    print(e)

## 6. Nested Models
Nesting models to validate complex, hierarchical structures.

In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    order_id: str
    items: list[Item]
    tax_rate: float = 0.08

# Instantiate using nested dicts
my_order = Order(
    order_id="ORD-2025",
    items=[
        {"name": "Keyboard", "price": 49.99},
        {"name": "Mouse", "price": 19.99}
    ]
)
print(my_order)

# Export to standard dictionary and JSON
print("Dictionary:", my_order.model_dump())
print("JSON:", my_order.model_dump_json())

## 7. Pydantic Settings
Using `pydantic_settings` to read, parse, and validate application configuration from environment variables.

In [ ]:
import os
from pydantic_settings import BaseSettings, SettingsConfigDict

# Simulate environment variables
os.environ["APP_API_KEY"] = "supersecretkey"
os.environ["APP_PORT"] = "8080"
os.environ["APP_DEBUG"] = "True"

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="app_")
    
    api_key: str
    port: int = 8000
    debug: bool = False

settings = AppSettings()
print("Loaded Settings:")
print("API Key:", settings.api_key)
print("Port:", settings.port, type(settings.port))
print("Debug:", settings.debug, type(settings.debug))